### Table Structure Evaluation — PubTables-1M

Đánh giá 3 model Table Structure (**`TableFormer ACCURATE`**, **`RapidTable — SLANet_plus`**, **`RapidTable — UniTable`**) trên **1.000 bảng test ngẫu nhiên (`seed=42`)** của **PubTables-1M** (cố định Layout = `DocLayout-YOLO`, OCR = `PP-OCRv6_medium`, Dynamic Image Scaling).

* **`TEDS` *(Tree-Edit-Distance-based Similarity)*:** Độ tương đồng cây HTML bảng trên cả cấu trúc (`<tr>`, `<td>`, `rowspan`, `colspan`) lẫn nội dung chữ trong ô.
* **`TEDS-Struct` *(Structure-only TEDS)*:** Chỉ đo độ chính xác thuần cấu trúc hàng/cột/gộp ô, tách khỏi sai số OCR.
* **`Matched %` & `Matched-only`:** Tỷ lệ bảng được bước Layout (`DocLayout-YOLO`) khoanh trúng và điểm `TEDS` / `TEDS-Struct` tính riêng trên các bảng khớp đó.

In [1]:
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import display

BASE_DIR = Path(".").resolve()
GT_PATH = BASE_DIR / "ground_truth.jsonl"
MODELS = [
    "TableFormer ACCURATE",
    "RapidTable — SLANet_plus",
    "RapidTable — UniTable",
]


def levenshtein(a: str, b: str) -> int:
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    if len(a) > len(b):
        a, b = b, a
    m = len(a)
    peq: dict[str, int] = {}
    for i, ch in enumerate(a):
        peq[ch] = peq.get(ch, 0) | (1 << i)
    pv = (1 << m) - 1
    mv = 0
    score = m
    hb = 1 << (m - 1)
    mask = (1 << m) - 1
    for ch in b:
        eq = peq.get(ch, 0)
        xv = eq | mv
        xh = (((eq & pv) + pv) ^ pv) | eq
        ph = mv | (~(xh | pv) & mask)
        mh = pv & xh
        if ph & hb:
            score += 1
        elif mh & hb:
            score -= 1
        ph = ((ph << 1) | 1) & mask
        mh = (mh << 1) & mask
        pv = mh | (~(xv | ph) & mask)
        mv = ph & xv
    return score


def norm_cell_text(s: str) -> str:
    s = unicodedata.normalize("NFKC", s or "")
    return re.sub(r"\s+", " ", s).strip()


class Node:
    __slots__ = ("tag", "colspan", "rowspan", "text", "children")

    def __init__(self, tag, colspan=1, rowspan=1, text="", children=None):
        self.tag = tag
        self.colspan = colspan
        self.rowspan = rowspan
        self.text = text
        self.children = children if children is not None else []


def _overlap_1d(a0: float, a1: float, b0: float, b1: float) -> float:
    inter = max(0.0, min(a1, b1) - max(a0, b0))
    denom = max(1e-6, min(a1 - a0, b1 - b0))
    return inter / denom


def pubtables_gt_to_tree(rec: dict) -> Node:
    """Reconstruct HTML table tree from PubTables-1M structure objects + words."""
    objs = rec.get("objects") or []
    row_boxes = sorted(
        [o["bbox"] for o in objs if o.get("label") == "table row"],
        key=lambda b: (b[1] + b[3]) / 2.0,
    )
    col_boxes = sorted(
        [o["bbox"] for o in objs if o.get("label") == "table column"],
        key=lambda b: (b[0] + b[2]) / 2.0,
    )
    if not row_boxes or not col_boxes:
        return Node("table")
    n_rows, n_cols = len(row_boxes), len(col_boxes)

    # Map spanning cells & projected row headers onto (r0, r1, c0, c1) grid ranges
    span_boxes = [
        o["bbox"] for o in objs
        if o.get("label") in {"table spanning cell", "table projected row header"}
    ]
    occupied = [[False] * n_cols for _ in range(n_rows)]
    cell_specs = []  # (r0, c0, rs, cs, bbox)

    for sb in span_boxes:
        r_idxs = [i for i, rb in enumerate(row_boxes) if _overlap_1d(sb[1], sb[3], rb[1], rb[3]) >= 0.5]
        c_idxs = [j for j, cb in enumerate(col_boxes) if _overlap_1d(sb[0], sb[2], cb[0], cb[2]) >= 0.5]
        if not r_idxs or not c_idxs:
            continue
        r0, r1 = min(r_idxs), max(r_idxs)
        c0, c1 = min(c_idxs), max(c_idxs)
        if r0 == r1 and c0 == c1:
            continue
        if any(occupied[r][c] for r in range(r0, r1 + 1) for c in range(c0, c1 + 1)):
            continue
        for r in range(r0, r1 + 1):
            for c in range(c0, c1 + 1):
                occupied[r][c] = True
        bbox = (col_boxes[c0][0], row_boxes[r0][1], col_boxes[c1][2], row_boxes[r1][3])
        cell_specs.append((r0, c0, r1 - r0 + 1, c1 - c0 + 1, bbox))

    for r in range(n_rows):
        for c in range(n_cols):
            if not occupied[r][c]:
                bbox = (col_boxes[c][0], row_boxes[r][1], col_boxes[c][2], row_boxes[r][3])
                cell_specs.append((r, c, 1, 1, bbox))

    # Assign GT words to cells by word center
    words = rec.get("words") or []
    cell_words: list[list[tuple[float, float, str]]] = [[] for _ in cell_specs]
    for w in words:
        txt = (w.get("text") or "").strip()
        wb = w.get("bbox")
        if not txt or not wb:
            continue
        cx = (wb[0] + wb[2]) / 2.0
        cy = (wb[1] + wb[3]) / 2.0
        for idx, (_, _, _, _, cb) in enumerate(cell_specs):
            if cb[0] <= cx <= cb[2] and cb[1] <= cy <= cb[3]:
                cell_words[idx].append((wb[1], wb[0], txt))
                break

    by_row: dict[int, list[tuple[int, Node]]] = {}
    for idx, (r0, c0, rs, cs, _) in enumerate(cell_specs):
        ws = sorted(cell_words[idx], key=lambda x: (round(x[0] / 4.0), x[1]))
        txt = norm_cell_text(" ".join(t for _, _, t in ws))
        by_row.setdefault(r0, []).append((c0, Node("td", cs, rs, txt)))

    rows = []
    for r_idx in sorted(by_row):
        row_cells = [n for _, n in sorted(by_row[r_idx], key=lambda x: x[0])]
        rows.append(Node("tr", children=row_cells))
    return Node("table", children=rows)


def docling_table_to_tree(tbl: dict) -> Node:
    data = tbl.get("data") or {}
    cells = data.get("table_cells") or []
    if not cells:
        return Node("table")
    by_row: dict[int, list[tuple[int, Node]]] = {}
    for c in cells:
        r0 = int(c.get("start_row_offset_idx", 0))
        c0 = int(c.get("start_col_offset_idx", 0))
        rs = max(1, int(c.get("row_span", 1)))
        cs = max(1, int(c.get("col_span", 1)))
        txt = norm_cell_text(c.get("text", ""))
        by_row.setdefault(r0, []).append((c0, Node("td", cs, rs, txt)))
    rows = []
    for r_idx in sorted(by_row):
        row_cells = [n for _, n in sorted(by_row[r_idx], key=lambda x: x[0])]
        rows.append(Node("tr", children=row_cells))
    return Node("table", children=rows)


def is_complex_tree(root: Node) -> bool:
    return any(c.colspan > 1 or c.rowspan > 1 for tr in root.children for c in tr.children)


def tree_size(root: Node) -> int:
    return 1 + sum(1 + len(tr.children) for tr in root.children)


def cell_rename_cost(a: Node, b: Node, struct_only: bool) -> float:
    if a.colspan != b.colspan or a.rowspan != b.rowspan:
        return 1.0
    if struct_only or a.text == b.text:
        return 0.0
    denom = max(len(a.text), len(b.text))
    return levenshtein(a.text, b.text) / denom if denom > 0 else 0.0


def row_ted(ra: Node, rb: Node, struct_only: bool) -> float:
    ca, cb = ra.children, rb.children
    m, n = len(ca), len(cb)
    prev = [float(j) for j in range(n + 1)]
    for i in range(1, m + 1):
        curr = [float(i)] + [0.0] * n
        ai = ca[i - 1]
        for j in range(1, n + 1):
            cost = cell_rename_cost(ai, cb[j - 1], struct_only)
            curr[j] = min(prev[j] + 1.0, curr[j - 1] + 1.0, prev[j - 1] + cost)
        prev = curr
    return prev[n]


def compute_teds(pred: Node, gt: Node, struct_only: bool = False) -> float:
    sa, sb = tree_size(pred), tree_size(gt)
    if sa == 1 and sb == 1:
        return 1.0
    if sa == 1 or sb == 1:
        return 0.0
    ra, rb = pred.children, gt.children
    m, n = len(ra), len(rb)
    prev = [0.0] * (n + 1)
    for j in range(1, n + 1):
        prev[j] = prev[j - 1] + 1.0 + len(rb[j - 1].children)
    for i in range(1, m + 1):
        del_cost = 1.0 + len(ra[i - 1].children)
        curr = [prev[0] + del_cost] + [0.0] * n
        rai = ra[i - 1]
        for j in range(1, n + 1):
            ins_cost = 1.0 + len(rb[j - 1].children)
            sub_cost = row_ted(rai, rb[j - 1], struct_only)
            curr[j] = min(prev[j] + del_cost, curr[j - 1] + ins_cost, prev[j - 1] + sub_cost)
        prev = curr
    return max(0.0, 1.0 - prev[n] / max(sa, sb))


# Collect the common set of predicted stems across models (all 1,000 sampled tables)
all_stems = set()
model_files: dict[str, dict[str, Path]] = {}
for m in MODELS:
    fmap = {
        f.stem: f for f in (BASE_DIR / m).rglob("*.json")
        if f.name not in {"errors.json", "current_files.json"}
    }
    model_files[m] = fmap
    all_stems.update(fmap.keys())

# Stream 2.4 GB ground_truth.jsonl and load only the 1,000 matched GT records
gt_trees: dict[str, Node] = {}
q = '"'
with open(GT_PATH, encoding="utf-8") as fh:
    for line in fh:
        parts = line[:120].split(q)
        if len(parts) >= 4 and parts[3] in all_stems:
            rec = json.loads(line)
            gt_trees[parts[3]] = pubtables_gt_to_tree(rec)
            if len(gt_trees) == len(all_stems):
                break

records = []
for model in MODELS:
    fmap = model_files[model]
    for stem, gt_tree in sorted(gt_trees.items()):
        if tree_size(gt_tree) <= 1:
            continue
        complexity = "Complex (Spanning)" if is_complex_tree(gt_tree) else "Simple (Grid)"
        f = fmap.get(stem)
        if f is None:
            records.append({"model": model, "file": stem, "complexity": complexity, "matched": False, "teds": 0.0, "teds_struct": 0.0})
            continue
        doc = json.loads(f.read_text(encoding="utf-8"))
        pr_tbls = doc.get("tables") or []
        if pr_tbls:
            # Pick the predicted table with the most cells (PubTables-1M has 1 table per PDF)
            best_tbl = max(pr_tbls, key=lambda t: len((t.get("data") or {}).get("table_cells") or []))
            pr_tree = docling_table_to_tree(best_tbl)
            matched = tree_size(pr_tree) > 1
            teds = compute_teds(pr_tree, gt_tree, False) * 100.0 if matched else 0.0
            teds_s = compute_teds(pr_tree, gt_tree, True) * 100.0 if matched else 0.0
        else:
            matched, teds, teds_s = False, 0.0, 0.0
        records.append({
            "model": model,
            "file": stem,
            "complexity": complexity,
            "matched": matched,
            "teds": teds,
            "teds_struct": teds_s,
        })

df = pd.DataFrame(records)
print(f"Evaluated {df['file'].nunique()} PubTables-1M tables x {len(MODELS)} models = {len(df)} rows")

Evaluated 1000 PubTables-1M tables x 3 models = 3000 rows


In [2]:
# Table 1: Overall Table Recognition Performance
rows = []
for model in MODELS:
    sub = df[df["model"] == model]
    m_sub = sub[sub["matched"]]
    rows.append({
        "Model": model,
        "Tables": len(sub),
        "Matched % (↑)": round(sub["matched"].mean() * 100.0, 2),
        "TEDS % (↑)": round(sub["teds"].mean(), 2),
        "TEDS-Struct % (↑)": round(sub["teds_struct"].mean(), 2),
        "TEDS Matched-only % (↑)": round(m_sub["teds"].mean(), 2) if len(m_sub) else 0.0,
        "TEDS-Struct Matched-only % (↑)": round(m_sub["teds_struct"].mean(), 2) if len(m_sub) else 0.0,
    })
overall_df = pd.DataFrame(rows).sort_values("TEDS % (↑)", ascending=False).reset_index(drop=True)
display(overall_df)

,Model,Tables,Matched % (↑),TEDS % (↑),TEDS-Struct % (↑),TEDS Matched-only % (↑),TEDS-Struct Matched-only % (↑)
0,RapidTable — UniTable,1000,98.0,86.55,90.61,88.31,92.46
1,TableFormer ACCURATE,1000,97.9,85.74,89.10,87.58,91.01
2,RapidTable — SLANet_plus,1000,98.0,85.28,91.33,87.02,93.20


In [3]:
# Table 2: Simple (Grid) vs Complex (Spanning) Tables
comp_rows = []
for comp in ["Simple (Grid)", "Complex (Spanning)"]:
    for model in MODELS:
        sub = df[(df["complexity"] == comp) & (df["model"] == model)]
        comp_rows.append({
            "Table Type": comp,
            "Model": model,
            "Tables": len(sub),
            "Matched % (↑)": round(sub["matched"].mean() * 100.0, 2),
            "TEDS % (↑)": round(sub["teds"].mean(), 2),
            "TEDS-Struct % (↑)": round(sub["teds_struct"].mean(), 2),
        })
display(pd.DataFrame(comp_rows))

,Table Type,Model,Tables,Matched % (↑),TEDS % (↑),TEDS-Struct % (↑)
0,Simple (Grid),TableFormer ACCURATE,482,96.06,88.52,91.65
1,Simple (Grid),RapidTable — SLANet_plus,482,96.27,88.87,93.88
2,Simple (Grid),RapidTable — UniTable,482,96.27,90.20,94.08
3,Complex (Spanning),TableFormer ACCURATE,518,99.61,83.15,86.73
4,Complex (Spanning),RapidTable — SLANet_plus,518,99.61,81.94,88.97
5,Complex (Spanning),RapidTable — UniTable,518,99.61,83.15,87.38
